# Qwen3.5-0.8B Department Router (Text-Only JSON Tags)

Fine-tune **Qwen3.5-0.8B** to route user queries into one of four tags:

- `visa`
- `residency`
- `traffic`
- `general`

The model must always respond with JSON only:

```json
{"department": "visa"}
```

**Example**
- User: `I want to apply for visa`
- Assistant: `{"department": "visa"}`

Workflow:
1. Generate synthetic data with `generate_routing_dataset.py` (Groq API)
2. Train LoRA in this notebook
3. Compare base vs fine-tuned with `compare_models.py`

### Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except:
        _numpy = "numpy"
        _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0 datasets
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
!uv pip install -qqq openai

### Optional: Generate synthetic dataset with Groq

Each training row only needs `query`, `department`, and JSON `response`.

```bash
python generate_routing_dataset.py --per-department 1000 --no-seed
```

This writes `data/routing_dataset.jsonl` with 1000 examples per department.

In [ ]:
# Uncomment to generate data inside Colab
# import os
# os.environ["GROQ_API_KEY"] = "YOUR_GROQ_KEY"
# !python generate_routing_dataset.py --per-department 1000

### Load model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-0.8B",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    load_in_16bit = True,
)

### Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = max_seq_length,
)

<a name="Data"></a>
### Data prep

Each training row becomes a chat conversation. The assistant always returns JSON routing tags only.

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

from routing_config import SYSTEM_PROMPT, build_messages, format_assistant_response

DATASET_PATH = Path("data/routing_dataset.jsonl")
SEED_PATH = Path("data/seed_routing_dataset.jsonl")

source_path = DATASET_PATH if DATASET_PATH.exists() else SEED_PATH
print(f"Loading dataset from: {source_path}")

raw_rows = []
with source_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_rows.append(json.loads(line))

def to_conversation(row):
    department = row["department"].strip().lower()
    return {
        "messages": build_messages(row["query"], department),
    }

converted_dataset = [to_conversation(row) for row in raw_rows]
print(f"Examples: {len(converted_dataset)}")
converted_dataset[0]

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize = False,
            add_generation_prompt = False,
            enable_thinking = False,
        )
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_list(converted_dataset)
dataset = dataset.map(formatting_prompts_func, batched = True)
dataset[0]["text"][:500]

### Pre-training inference check

In [ ]:
from routing_config import parse_department_tag
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

sample_query = "I want to apply for visa"
messages = build_messages(sample_query)
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,
)

inputs = tokenizer(prompt, return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 32, use_cache = True, temperature = 0.0, do_sample = False)
generated = tokenizer.decode(outputs[0, inputs["input_ids"].shape[1]:], skip_special_tokens = True)
print("Raw:", generated)
print("Parsed tag:", parse_department_tag(generated))

<a name="Train"></a>
### Train

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 120,
        # num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_router",
        report_to = "none",
        dataset_text_field = "text",
    ),
)

trainer_stats = trainer.train()
trainer_stats.metrics

<a name="Inference"></a>
### Inference

In [ ]:
def route_query(query: str) -> dict:
    FastLanguageModel.for_inference(model)
    messages = build_messages(query)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True,
        enable_thinking = False,
    )
    inputs = tokenizer(prompt, return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 32, use_cache = True, temperature = 0.0, do_sample = False)
    raw = tokenizer.decode(outputs[0, inputs["input_ids"].shape[1]:], skip_special_tokens = True).strip()
    tag = parse_department_tag(raw)
    return {"department": tag, "raw": raw}

for q in [
    "I want to apply for visa",
    "What are your office hours?",
    "The camera caught me yesterday.",
]:
    print(q, "->", route_query(q))

### Evaluate on 15 tricky routing questions

In [ ]:
import json

with open("test_questions.json", "r", encoding="utf-8") as f:
    test_questions = json.load(f)

correct = 0
print(f"{'ID':<4} {'Expected':<10} {'Predicted':<10} OK   Query")
print("-" * 90)
for item in test_questions:
    result = route_query(item["query"])
    predicted = result["department"]
    ok = predicted == item["expected"]
    correct += int(ok)
    print(f"{item['id']:<4} {item['expected']:<10} {str(predicted):<10} {str(ok):<4} {item['query'][:55]}")

print(f"\nAccuracy: {correct}/{len(test_questions)} ({correct/len(test_questions):.1%})")

<a name="Save"></a>
### Save LoRA adapters

In [ ]:
LORA_DIR = "qwen_router_lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"Saved LoRA adapters to {LORA_DIR}")

### Compare base vs fine-tuned (local script)

After saving LoRA adapters, run:

```bash
python compare_models.py --lora-path qwen_router_lora
```